In [ ]:
import mlflow
import pandas as pd
from dotenv import load_dotenv
from mlflow import MlflowClient
from mlflow.deployments import get_deploy_client

mlflow.set_registry_uri("databricks-uc")

In [ ]:
client_mlflow = MlflowClient()
client_databricks = get_deploy_client("databricks")

In [ ]:
model_version = client_mlflow.get_model_version_by_alias(
    "betsim.models.xgboost_optuna",
    "champion",
)
version = model_version.version

In [ ]:
endpoints = client_databricks.list_endpoints()
is_exist = any(e["name"] == "betsim" for e in endpoints)

In [ ]:
if not is_exist:
    endpoint = client_databricks.create_endpoint(
        config={
            "name": "betsim",
            "config": {
                "served_entities": [
                    {
                        "entity_name": "betsim.models.xgboost_optuna",
                        "entity_version": version,
                        "workload_size": "Small",
                        "workload_type": "CPU",
                        "scale_to_zero_enabled": True,
                    },
                ],
            },
        },
    )
    print("Endpoint created.")
else:
    print("Endpoint already exists.")